# Ejercicio 5

Siguiendo el trabajo de Hinton y Salakhutdinov (2006), entrene una máquina restringida de Boltzmann con imágenes de la base de datos MNIST. Muestre el error de recontruccion durante el entrenamiento, y ejemplos de cada uno de los dígitos reconstruidos.

In [ ]:
import itertools
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_openml

In [ ]:
np.random.seed(110007)

In [ ]:
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist["data"].reshape(-1, 784).astype(float) / 255.0, mnist["target"]

idx = np.random.permutation(X.shape[0])
train_size = 60000
train_idx, test_idx = idx[:train_size], idx[train_size:]
X_train, y_train, X_test, y_test = X[train_idx, :], y[train_idx], X[test_idx, :], y[test_idx]

In [ ]:
class RBM:
    def __init__(self, n_visible, n_hidden):
        self.weights = np.random.normal(0, 0.01, size=(n_visible, n_hidden))
        self.visible_biases = np.random.normal(0, 0.01, size=(1, n_visible))
        self.hidden_biases = np.random.normal(0, 0.01, size=(1, n_hidden))
        self.train_errors = []
        self.test_errors = []

    def visible_to_hidden(self, V, sample=True):
        prob_h = self.sigmoid(np.dot(V, self.weights) + self.hidden_biases)
        H = (np.random.random(size=prob_h.shape) < prob_h).astype(float) if sample else None
        return prob_h, H 

    def hidden_to_visible(self, H, sample=True):
        prob_v = self.sigmoid(np.dot(H, self.weights.T) + self.visible_biases)
        V = (np.random.random(size=prob_v.shape) < prob_v).astype(float) if sample else None
        return prob_v, V

    def train(self, X, X_test=None, epochs=10, batch_size=1, lr=0.1, print_errors=False):
        n_samples = X.shape[0]
        for e in range(epochs):
            idx = np.random.permutation(n_samples)
            X_shuf = X[idx]
            train_err = 0.0
            n_batches = 0
            for i in range(0, n_samples, batch_size):
                V0 = X_shuf[i : i + batch_size]
                curr_batch_size = V0.shape[0]

                prob_h0, H0 = self.visible_to_hidden(V0)
                prob_v1, _ = self.hidden_to_visible(H0, sample=False)
                V1 = prob_v1
                prob_h1, _ = self.visible_to_hidden(prob_v1, sample=False)
                
                data_correction = np.dot(V0.T, prob_h0) / curr_batch_size
                confab_correction = np.dot(V1.T, prob_h1) / curr_batch_size
                
                dW = lr * (data_correction - confab_correction)
                db_v = lr * np.mean(V0 - V1, axis=0, keepdims=True)
                db_h = lr * np.mean(prob_h0 - prob_h1, axis=0, keepdims=True)
        
                self.weights += dW
                self.visible_biases += db_v
                self.hidden_biases  += db_h

                train_err += np.mean((V0 - self.reconstruct(V0)) ** 2)
                n_batches += 1

            train_err /= n_batches
            self.train_errors.append(train_err)
            if X_test is not None:
                self.test_errors.append(np.mean((X_test - self.reconstruct(X_test)) ** 2))
            if print_errors:
                print(f"Epoch {e+1:3d} - train error: {train_err:.6f}")

    def reconstruct(self, X):
        prob_h0, _ = self.visible_to_hidden(X, sample=False)
        prob_v1, _ = self.hidden_to_visible(prob_h0, sample=False)
        return prob_v1
    
    @staticmethod
    def sigmoid(X):
        return 1 / (1 + np.exp(-np.clip(X, -500, 500)))

In [ ]:
def draw_originals_and_reconstructions(originals, reconstructions, hidden_sizes, path=None):
    num_imgs = originals.shape[0]
    num_reconstructions = len(reconstructions)
    plt.figure(figsize=(num_imgs * 2, 2 + num_reconstructions * 2))
    for i in range(num_imgs):
        ax1 = plt.subplot(1 + num_reconstructions, num_imgs, i + 1)
        plt.imshow(originals[i,:].reshape(28, 28), vmin=0, vmax=1, cmap='grey')
        plt.tight_layout()
        axs = []
        for j in range(num_reconstructions):
            axs.append(plt.subplot(1 + num_reconstructions, num_imgs, i + (j + 1) * num_imgs + 1))
            plt.imshow(reconstructions[j][i,:].reshape(28, 28), vmin=0, vmax=1, cmap='grey')
            
        if i == 0:
            ax1.set_ylabel('Original')
            ax1.tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
            for j in range(num_reconstructions):
                axs[j].set_ylabel(f'Reconstrucción {hidden_sizes[j]}')
                axs[j].tick_params(axis='both', which='both', bottom=False, top=False, left=False, right=False, labelbottom=False, labelleft=False)
        else:
            ax1.axis('off')
            for ax in axs:
                ax.axis('off')

    plt.tight_layout()
    if path is not None:
        plt.savefig(path)
    plt.show()

In [ ]:
def plot_errors(model, path=None):
    plt.figure(figsize=(6,4))
    plt.plot(model.train_errors, label='Error de entrenamiento')
    plt.plot(model.test_errors, label='Error de evaluación')
    plt.grid()
    plt.legend()
    plt.xlabel('Época')
    plt.ylabel('Error de reconstrucción')
    plt.tight_layout()
    if path is not None:
        plt.savefig(path)
    plt.show()

In [ ]:
hidden_sizes = [50, 20, 2]
learning_rates = [0.1, 0.05, 0.01]
reconstructions = []

for num_hidden, lr in zip(hidden_sizes, learning_rates):
    print(f"\n==== Hidden layer size: {num_hidden} ====")
    model = RBM(784, num_hidden)
    model.train(X_train, X_test=X_test, batch_size=100, epochs=10, lr=lr)
    
    test_reconstructions = model.reconstruct(X_test)
    reconstructions.append(test_reconstructions)
    test_error = np.mean((X_test - test_reconstructions) ** 2)
    print(f"Train error: {model.train_errors[-1]:.4f} - Test error: {test_error:.4f}")
    plot_errors(model, path=f'../informe/img/ej5/error_{num_hidden}.svg')

draw_originals_and_reconstructions(X_test[:5, :], reconstructions, hidden_sizes, path='../informe/img/ej5/reconstructions.svg')